# Step 6: The final run (test chats opened once)
### Neeraj, MMAI 894 (router fine-tuning)

**What happens here, once:** the 40 locked test chats and the team's 31 fresh chats are opened for the first and only time, and every router is scored on them by the same code:

| Router | What it is |
|---|---|
| Haiku 5.5 | the team's router today (reference) |
| Sonnet 5.5 | the large prompted model |
| Jev, same information | Jev given the same rules and examples as Haiku (notebook 04b) |
| Jev, first version | for completeness |
| **Fine-tuned, safest** | full fine-tuning + attack answer: misses fewest people on the practice chats, but hands many ordinary chats to staff |
| **Fine-tuned, balanced** | LoRA + attack answer + its own unsure line: best overall of the fine-tuned options on the practice chats |

**Nothing is chosen here.** Every setting (epochs, cut-offs, unsure lines, seeds, models, repeats) was frozen in `config/final_settings.json` from the practice chats before this notebook was written. The fine-tuned models are trained on all 160 practice chats with 5 seeds; the test and fresh chats are scored only after the last epoch, and their answers are never given to training.

**What you need to do:**
1. Accelerator **GPU T4**, Internet **On**, and the `OPENROUTER_API_KEY` secret ticked for this notebook.
2. Make sure OpenRouter has at least **$5 of credit** and that the key's own limit leaves at least $5 (expected cost about $3; the run stops itself at $6).
3. Run all cells: about 30 to 40 minutes. If the session stops, run again: finished training runs and saved answers are reused.
4. Download `results_06.zip` (directly in `/kaggle/working`) and the notebook into `results/step06`.

Everything shown is computed from the runs; this text holds no results.

## Step 0: Get the latest code

In [1]:
import importlib
import os
import shutil
import sys

REPO_URL = "https://github.com/neerajmehla28/northstar-finetuning"
REPO_DIR = "/kaggle/working/northstar-finetuning"

%cd /kaggle/working
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -q {REPO_URL} {REPO_DIR}
!pip install -q peft
!pip uninstall -y -q torchao                 # Kaggle's older torchao makes the new peft refuse to run; we do not use it

%cd {REPO_DIR}
sys.path.insert(0, "src")
importlib.invalidate_caches()
for name in [m for m in sys.modules if m.startswith("northstar")]:
    del sys.modules[name]

for need in ("config/final_settings.json", "src/northstar/finetune/final_run.py", "data/fresh/fresh_chats.csv"):
    assert os.path.exists(need), f"missing {need}: has everything been pushed to GitHub?"
!git log --oneline -3

/kaggle/working
/kaggle/working/northstar-finetuning
21a74de (HEAD -> main, origin/main, origin/HEAD) Add the final run (notebook 06), frozen settings, fresh held-out set; second notebook 05 run
65eee80 Add Jev version 2 results, desk check and unsure-line rule, updated notebook 05
214fae1 Add Jev version 2 (same information as Haiku), notebook 04b, 03c and 05 results, comparison tables


## Step 1: Machine, key and the frozen settings

In [2]:
import json
import time

import numpy as np
import pandas as pd
import torch
from kaggle_secrets import UserSecretsClient

from northstar.finetune import baselines as B
from northstar.finetune import final_run as F
from northstar.finetune import protocol as P
from northstar.finetune import router_options as O
from northstar.finetune import routing as R

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), "No GPU found. Set Accelerator to GPU T4."
try:
    API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
except Exception as e:
    raise RuntimeError("No secret named OPENROUTER_API_KEY attached to this notebook: Add-ons -> Secrets, tick it, run this cell again.") from e

S = F.load_settings()
print("Settings frozen on", S["frozen_on"], "from", S["frozen_from"])
for name, s in S["fine_tuned"].items():
    print(f"  {name:9s} {s['label']}: input {s['input']}, epochs {s['training']['epochs']}, cut-offs escalation {s['escalation_cutoff']} / "
          f"attack {s['attack_cutoff']}, unsure line {s['unsure_line']}")
print("  language models:", {m: v["repeats"] for m, v in S["language_models"].items()}, "| inputs:", S["language_model_inputs"])

RESULTS = "/kaggle/working/results_06"
RUNS, PREDS, CACHE = f"{RESULTS}/fine_tuned", f"{RESULTS}/preds", f"{RESULTS}/cache"
os.makedirs(RESULTS, exist_ok=True)
shutil.copy("config/final_settings.json", f"{RESULTS}/final_settings.json")
client = B.Client(API_KEY, CACHE, budget_usd=S["api_budget_usd"])
pool = P.pool(P.load_all())

torch 2.11.0+cu130 | GPU: Tesla T4
Settings frozen on 2026-11-26 from results/step05 (notebook 05, second run, commit 65eee80) and results/cv_step03c, before any test or fresh chat was opened
  safest    Fine-tuned, full fine-tuning + attack answer: input first_message, epochs 27, cut-offs escalation 0.05 / attack 0.05, unsure line 0.55
  balanced  Fine-tuned, LoRA + attack answer + own unsure line: input first_message, epochs 60, cut-offs escalation 0.1 / attack 0.05, unsure line 0.4
  language models: {'haiku': 3, 'sonnet': 3, 'jev2': 2, 'jev': 2} | inputs: ['first_message', 'full_conversation']


## Step 2: Open the test and fresh chats (once)
Both sets are locked in the code; opening them needs `final=True` and an environment switch, and each opening is written to `outputs/test_access.log`.

In [3]:
os.environ["NORTHSTAR_ALLOW_TEST"] = "1"
test = P.test_set(P.load_all(), final=True, note="final run, notebook 06, settings frozen in config/final_settings.json")
fresh = P.fresh_set(final=True, note="final run, notebook 06, settings frozen in config/final_settings.json")
held = pd.concat([test.assign(dataset="test"), fresh.assign(dataset="fresh")], ignore_index=True)
for name, d in (("test", test), ("fresh", fresh)):
    print(f"{name:5s}: {len(d)} chats, {int(d['escalation_label'].sum())} need a person, {int(d['attack_label'].sum())} attacks, "
          f"{int((d['language'] == 'fr').sum())} French, {d['customer_id'].nunique()} customers")
assert not set(held["id"]) & set(pool["id"]), "a held-out chat is also in the practice pool"
print(open("outputs/test_access.log").read())

test : 40 chats, 18 need a person, 9 attacks, 14 French, 20 customers
fresh: 31 chats, 24 need a person, 9 attacks, 10 French, 20 customers
2026-11-26T19:33:27 test set opened: final run, notebook 06, settings frozen in config/final_settings.json
2026-11-26T19:33:27 fresh set opened: final run, notebook 06, settings frozen in config/final_settings.json



## Step 3: Smoke test (practice chats only)
One seed, one epoch, scoring 8 practice chats: proves the training code runs before the real training. No held-out chat is used here.

In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
for name, s in S["fine_tuned"].items():
    cfg = F.train_config({**s["training"], "epochs": 1})
    t0 = time.time()
    F.train_and_score(cfg, s["input"], pool, pool.head(8).assign(dataset="smoke"), [1], f"/kaggle/working/smoke/{name}",
                      tokenizer=tokenizer, verbose=False)
    print(f"{name:9s} smoke test OK: {time.time() - t0:.0f}s for 1 epoch")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

... (2 outputs left out)

safest    smoke test OK: 14s for 1 epoch


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


balanced  smoke test OK: 3s for 1 epoch


## Step 4: Train both fine-tuned setups on all 160 practice chats, score the held-out chats once
Five seeds each. The LoRA setup's trained add-ons and heads are saved (a few MB per seed) for the hand-over; the full fine-tuning weights (1.1 GB each) are not. One model of each setup is kept to measure its speed on a CPU (2 cores, one chat at a time, like the team's server).

In [5]:
SEEDS = S["fine_tuned_common"]["seeds"]
paths, latency = {}, {}
first_messages = list(P.input_text(test, "first_message"))
for name, s in S["fine_tuned"].items():
    print(f"=== {name}: {s['label']}")
    cfg = F.train_config(s["training"])
    paths[name], model = F.train_and_score(cfg, s["input"], pool, held, SEEDS, f"{RUNS}/{name}", keep_seed=SEEDS[0],
                                           save_trained_weights=cfg.lora, tokenizer=tokenizer)
    latency[name] = F.cpu_latency(model, tokenizer, first_messages, cfg.max_length, threads=2)
    print(f"  CPU speed, one model: median {latency[name]['median_s']:.3f}s, p90 {latency[name]['p90_s']:.3f}s per chat")
    del model
    torch.cuda.empty_cache()
pd.DataFrame(latency).T.to_csv(f"{RESULTS}/fine_tuned_cpu_latency.csv")

=== safest: Fine-tuned, full fine-tuning + attack answer


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  seed 101: trained on 160 chats in 54s, scored 71 held-out chats once


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (21 outputs left out)

  seed 104: trained on 160 chats in 55s, scored 71 held-out chats once


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  seed 105: trained on 160 chats in 55s, scored 71 held-out chats once
  CPU speed, one model: median 0.153s, p90 0.171s per chat


## Step 5: The language models on the same chats
Same prompt, settings and repeats as on the practice chats. A probe of 3 chats per model first, with a cost estimate, then the full run. Saved answers are reused if the cell is run again.

In [6]:
LLMS = S["language_models"]
INPUTS = S["language_model_inputs"]
probe = pd.concat([B.run_variant(client, m, held, "first_message", 0, PREDS, limit=3) for m in LLMS], ignore_index=True)
assert not probe["fallback"].any(), "a probe call gave no usable answer: see the error column"
per_chat = probe.groupby("model")["cost_usd"].mean()
estimate = sum(per_chat[m] * len(held) * len(INPUTS) * v["repeats"] for m, v in LLMS.items()) * 1.15
print("cost per chat:", per_chat.round(6).to_dict(), f"| estimated: ${estimate:.2f} (limit ${S['api_budget_usd']:.0f})")
assert estimate < S["api_budget_usd"], "estimate above the limit"

for m, v in LLMS.items():
    for inp in INPUTS:
        for rep in range(v["repeats"]):
            d = B.run_variant(client, m, held, inp, rep, PREDS, workers=3)
            print(f"{m:6s} {inp:17s} repeat {rep}: fallbacks {int(d['fallback'].sum())} | ${d['cost_usd'].sum():.3f}", flush=True)
print(f"New money spent in this session: ${client.spent:.2f}")

cost per chat: {'haiku': 0.000292, 'jev': 7.3e-05, 'jev2': 0.000247, 'sonnet': 0.005183} | estimated: $2.79 (limit $6)
haiku  first_message     repeat 0: fallbacks 0 | $0.023
haiku  first_message     repeat 1: fallbacks 0 | $0.024
haiku  first_message     repeat 2: fallbacks 0 | $0.023
haiku  full_conversation repeat 0: fallbacks 0 | $0.026
haiku  full_conversation repeat 1: fallbacks 0 | $0.026
haiku  full_conversation repeat 2: fallbacks 0 | $0.026
sonnet first_message     repeat 0: fallbacks 0 | $0.383
sonnet first_message     repeat 1: fallbacks 0 | $0.383
sonnet first_message     repeat 2: fallbacks 0 | $0.386
sonnet full_conversation repeat 0: fallbacks 0 | $0.406
sonnet full_conversation repeat 1: fallbacks 0 | $0.405
sonnet full_conversation repeat 2: fallbacks 0 | $0.404
jev2   first_message     repeat 0: fallbacks 0 | $0.018
jev2   first_message     repeat 1: fallbacks 0 | $0.018
jev2   full_conversation repeat 0: fallbacks 0 | $0.018
jev2   full_conversation repeat 1: fallba

## Step 6: The final scores
For each set (test, fresh) and input, every router scored the same way: safety first (missed, right team, attacks to the right team), then efficiency (extra handoffs, kept with the wrong desk), then the strict handled-right with its 95% range (resampling customers), and an exact paired test against Haiku on the same chats. The fine-tuned routers read the first message; their row is compared with Haiku on the first message.

In [7]:
NAMES = {"haiku": "Haiku (team's router)", "sonnet": "Sonnet", "jev2": "Jev, same information as Haiku", "jev": "Jev, first version",
         "safest": "Fine-tuned, safest", "balanced": "Fine-tuned, balanced"}
LAT = pd.read_csv("results/step04/latency.csv").set_index("model")["median_ms"] / 1000
LAT = pd.concat([LAT, pd.read_csv("results/step04b/latency.csv").set_index("model")["median_ms"] / 1000])
rows, routed, pairs, spread = [], {}, [], []
for dname in ("test", "fresh"):
    mask = (held["dataset"] == dname).values
    d = held[mask].reset_index(drop=True)
    for inp in INPUTS:
        for m in LLMS:
            reps = [r[mask].reset_index(drop=True) for r in B.load_preds(PREDS, m, inp, held)]
            llm = O.combine_llm(reps)
            lang = np.where(np.mean([(r["language"] == "fr").values for r in reps], axis=0) > 0.5, "fr", "en")
            row, rt = F.score_router(d, O.option_llm_alone(llm), lang)
            rows.append(dict(dataset=dname, input=inp, router=NAMES[m], seconds=LAT.get(m, np.nan), **row))
            routed[(dname, inp, m)] = rt["handled_right"].values
    for name, s in S["fine_tuned"].items():
        combined, per_seed, seeds, secs = F.tuned_answers(paths[name], held, s["escalation_cutoff"], s["attack_cutoff"])
        lang = P.input_text(d, s["input"]).map(R.language_rule).values
        row, rt = F.score_router(d, O.option_attack_head(combined[mask].reset_index(drop=True)), lang, s["unsure_line"])
        rows.append(dict(dataset=dname, input=s["input"], router=NAMES[name], seconds=latency[name]["median_s"], **row))
        routed[(dname, s["input"], name)] = rt["handled_right"].values
        for seed, a in zip(seeds, per_seed):
            r1, _ = F.score_router(d, O.option_attack_head(a[mask].reset_index(drop=True)), lang, s["unsure_line"], n_boot=10)
            spread.append(dict(dataset=dname, router=NAMES[name], seed=seed, type_accuracy=r1["type_accuracy"],
                               handled_right=r1["handled_right"], missed=r1["missed"], extra_handoffs=r1["extra_handoffs"]))
    for (dn, inp, key), hr in routed.items():
        if dn == dname and key != "haiku":
            pairs.append(dict(dataset=dname, input=inp, router=NAMES[key], versus="Haiku", **F.paired_vs(routed[(dname, inp, "haiku")], hr)))

final = pd.DataFrame(rows)
final.to_csv(f"{RESULTS}/final_scores.csv", index=False)
pd.DataFrame(pairs).to_csv(f"{RESULTS}/final_paired_vs_haiku.csv", index=False)
spread = pd.DataFrame(spread)
spread.to_csv(f"{RESULTS}/final_fine_tuned_by_seed.csv", index=False)

show = final.copy()
show["type right %"] = (100 * show["type_accuracy"]).round(1)
show["handled right % [95% range]"] = [f"{100 * h:.1f} [{100 * lo:.0f}-{100 * hi:.0f}]" for h, lo, hi in zip(show["handled_right"], show["handled_ci_low"], show["handled_ci_high"])]
show["needs a person: missed"] = show["missed"].astype(str) + " of " + show["needs_person"].astype(str)
show["attacks to right team"] = show["attacks_right_team"].astype(str) + " of " + show["attacks"].astype(str)
cols = ["router", "type right %", "needs a person: missed", "right_team", "attacks to right team", "extra_handoffs", "kept_wrong_desk",
        "handled right % [95% range]", "usd_per_1000", "seconds"]
for (dn, inp), g in show.groupby(["dataset", "input"], sort=False):
    print(f"\n===== {dn} chats, {inp.replace('_', ' ')}")
    display(g.sort_values("handled_right", ascending=False)[cols].set_index("router"))
display(pd.DataFrame(pairs).round(4))


===== test chats, first message


,type right %,needs a person: missed,right_team,attacks to right team,extra_handoffs,kept_wrong_desk,handled right % [95% range],usd_per_1000,seconds
router,,,,,,,,,
Haiku (team's router),82.5,0 of 18,18,9 of 9,2,2,90.0 [79-98],0.323403,2.340000
Sonnet,82.5,0 of 18,17,9 of 9,0,3,90.0 [79-98],5.374800,1.004000
"Jev, same information as Haiku",90.0,0 of 18,17,9 of 9,4,0,87.5 [76-97],0.248170,0.456000
"Jev, first version",77.5,0 of 18,16,7 of 9,3,0,87.5 [75-97],0.074080,0.200000
"Fine-tuned, balanced",80.0,2 of 18,15,8 of 9,1,1,87.5 [77-96],0.000000,0.152659
"Fine-tuned, safest",82.5,2 of 18,15,8 of 9,5,1,77.5 [64-90],0.000000,0.149729



===== test chats, full conversation


,type right %,needs a person: missed,right_team,attacks to right team,extra_handoffs,kept_wrong_desk,handled right % [95% range],usd_per_1000,seconds
router,,,,,,,,,
Haiku (team's router),82.5,0 of 18,18,9 of 9,3,2,87.5 [73-98],0.358475,2.340
Sonnet,77.5,1 of 18,16,9 of 9,2,3,82.5 [69-94],5.572250,1.004
"Jev, same information as Haiku",82.5,2 of 18,15,8 of 9,3,1,82.5 [70-93],0.252330,0.456
"Jev, first version",77.5,2 of 18,11,4 of 9,5,0,70.0 [54-86],0.078240,0.200



===== fresh chats, first message


,type right %,needs a person: missed,right_team,attacks to right team,extra_handoffs,kept_wrong_desk,handled right % [95% range],usd_per_1000,seconds
router,,,,,,,,,
"Jev, first version",87.1,6 of 24,17,8 of 9,0,0,77.4 [58-92],0.072813,0.200000
Sonnet,90.3,7 of 24,16,9 of 9,0,0,74.2 [53-89],5.445462,1.004000
"Jev, same information as Haiku",87.1,7 of 24,16,8 of 9,0,0,74.2 [54-89],0.246903,0.456000
Haiku (team's router),87.1,8 of 24,15,8 of 9,0,0,71.0 [47-89],0.341171,2.340000
"Fine-tuned, balanced",54.8,9 of 24,10,5 of 9,1,0,51.6 [36-67],0.000000,0.152659
"Fine-tuned, safest",58.1,4 of 24,11,6 of 9,4,0,45.2 [29-61],0.000000,0.149729



===== fresh chats, full conversation


,type right %,needs a person: missed,right_team,attacks to right team,extra_handoffs,kept_wrong_desk,handled right % [95% range],usd_per_1000,seconds
router,,,,,,,,,
Haiku (team's router),83.9,6 of 24,17,8 of 9,0,0,77.4 [57-93],0.379315,2.340
Sonnet,80.6,6 of 24,16,9 of 9,0,0,74.2 [54-89],5.876946,1.004
"Jev, same information as Haiku",80.6,6 of 24,15,7 of 9,0,0,71.0 [48-88],0.252417,0.456
"Jev, first version",77.4,7 of 24,12,5 of 9,0,0,61.3 [43-77],0.078327,0.200


,dataset,input,router,versus,only_router_right,only_reference_right,p_value
0,test,first_message,Sonnet,Haiku,2,2,1.0000
1,test,first_message,"Jev, same information as Haiku",Haiku,1,2,1.0000
2,test,first_message,"Jev, first version",Haiku,2,3,1.0000
3,test,full_conversation,Sonnet,Haiku,2,4,0.6875
4,test,full_conversation,"Jev, same information as Haiku",Haiku,3,5,0.7266
5,test,full_conversation,"Jev, first version",Haiku,2,9,0.0654
6,test,first_message,"Fine-tuned, safest",Haiku,1,6,0.1250
7,test,first_message,"Fine-tuned, balanced",Haiku,2,3,1.0000
8,fresh,first_message,Sonnet,Haiku,2,1,1.0000
9,fresh,first_message,"Jev, same information as Haiku",Haiku,1,0,1.0000


## Step 7: How steady are the fine-tuned models, and did the practice chats predict the test?
Each of the 5 seeds scored on its own (the combined router averages them), and the practice-chat estimate frozen in the settings next to the test and fresh results.

In [8]:
display(spread.groupby(["dataset", "router"])[["type_accuracy", "handled_right", "missed", "extra_handoffs"]].agg(["mean", "std"]).round(3))
check = []
for name, s in S["fine_tuned"].items():
    e = s["pool_estimate"]
    check.append(dict(router=NAMES[name], where="practice chats (estimate)", handled_right=e["handled_right"], missed=e["missed"], extra_handoffs=e["extra_handoffs"]))
    for dn in ("test", "fresh"):
        r = final[(final["router"] == NAMES[name]) & (final["dataset"] == dn)].iloc[0]
        check.append(dict(router=NAMES[name], where=f"{dn} chats", handled_right=round(r["handled_right"], 3), missed=r["missed"], extra_handoffs=r["extra_handoffs"]))
pd.DataFrame(check)

type_accuracy        handled_right        missed  \
                                      mean    std          mean    std   mean   
dataset router                                                                  
fresh   Fine-tuned, balanced         0.510  0.058         0.445  0.042    7.8   
        Fine-tuned, safest           0.523  0.027         0.477  0.074    4.8   
test    Fine-tuned, balanced         0.750  0.047         0.835  0.052    1.6   
        Fine-tuned, safest           0.750  0.081         0.755  0.037    0.8   

                                    extra_handoffs         
                                std           mean    std  
dataset router                                             
fresh   Fine-tuned, balanced  0.837            1.8  1.304  
        Fine-tuned, safest    0.837            4.0  0.707  
test    Fine-tuned, balanced  0.548            1.8  0.447  
        Fine-tuned, safest    0.837            6.2  1.789

,router,where,handled_right,missed,extra_handoffs
0,"Fine-tuned, safest",practice chats (estimate),0.644,2,45
1,"Fine-tuned, safest",test chats,0.775,2,5
2,"Fine-tuned, safest",fresh chats,0.452,4,4
3,"Fine-tuned, balanced",practice chats (estimate),0.794,6,15
4,"Fine-tuned, balanced",test chats,0.875,2,1
5,"Fine-tuned, balanced",fresh chats,0.516,9,1


## Step 8: Recordings for the team's end-to-end test
The team's harness can replay a router's recorded answers (`RecordedRouter` in `evaluation/setups.py`). These files hold each fine-tuned router's answers on the test and fresh chats in that format: type, escalation, reason (attack -> adversarial, else from the type), language (word rule) and a confidence rescaled so that the team's 0.55 unsure line acts as the router's own line.

In [9]:
os.makedirs(f"{RESULTS}/recordings", exist_ok=True)
for name, s in S["fine_tuned"].items():
    combined, _, _, _ = F.tuned_answers(paths[name], held, s["escalation_cutoff"], s["attack_cutoff"])
    lang = P.input_text(held, s["input"]).map(R.language_rule).values
    rec = F.recording(held, combined, lang, s["unsure_line"], 1000 * latency[name]["median_s"], tokenizer)
    rec.to_csv(f"{RESULTS}/recordings/router_finetuned-{name}_test_and_fresh.csv", index=False)
    print(name, "recording:", len(rec), "chats")

safest recording: 71 chats
balanced recording: 71 chats


## Step 9: Save the results
Download `results_06.zip` and the notebook (File -> Download Notebook) into `results/step06`.

In [10]:
shutil.copy("outputs/test_access.log", f"{RESULTS}/test_access.log")
shutil.make_archive("/kaggle/working/results_06", "zip", RESULTS)
print(f"results_06.zip {os.path.getsize('/kaggle/working/results_06.zip') / 1e6:.1f} MB")
for name in sorted(os.listdir(RESULTS)):
    print("  ", name)

results_06.zip 6.4 MB
   cache
   final_fine_tuned_by_seed.csv
   final_paired_vs_haiku.csv
   final_scores.csv
   final_settings.json
   fine_tuned
   fine_tuned_cpu_latency.csv
   preds
   recordings
   test_access.log


## What this run shows, and its limits
- **Shows:** the honest, once-only comparison of every router on chats nobody tuned on: the 40 test chats (same generator as the training data) and the 31 fresh chats (another model family, reviewed by two people, labelled under the team's final policy).
- **Limits:**
  - Small sets: 40 and 31 chats, so the 95% ranges are wide; differences of a few chats are not proof.
  - The fresh chats follow the team's final policy, which is stricter on some escalations than the data the models learned from; every router meets that difference, the fine-tuned ones most directly.
  - The team's own Haiku and Sonnet test runs (single runs, full conversation) used the same prompt; this run repeats them on both inputs with the same code as every other router.
  - The fine-tuned speed is one model on 2 CPU cores; the combined five-seed router takes about five times as long unless one seed is used.
  - The router alone is scored here; the team's whole-agent test (graded replies) is the next step and uses the recordings saved above.